In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
import numpy as np
import popsim.param_utils as param_utils
from popsim.modules.tearing import DisruptionPhase, IslandRotationPhase, Island, Tearing
from popsim.simulate import simulate

dt = 1e-4 / 3  # s
time_base = param_utils.make_time_base(t0=0.0, t1=7.0, dt=dt)
config = Tearing.Config(
    magx_time=time_base, thincurr_file="21_mode_resp_data.txt", ods_file="thatfile.txt"
)


def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]


def generate_disruption_phase_trajectory(trigger_time: float, tq_to_cq_dur: float):
    # TODO(allenw): we want a rectilinear interpolation scheme.
    disrupt_phase_dict = {
        0.0: DisruptionPhase.NONE,
        trigger_time - dt: DisruptionPhase.NONE,
        trigger_time: DisruptionPhase.TQ,
        trigger_time + tq_to_cq_dur - dt: DisruptionPhase.TQ,
        trigger_time + tq_to_cq_dur: DisruptionPhase.CQ,
    }

    # Round the times to the nearest time step in the time base.
    disrupt_phase_dict = {
        find_nearest(time_base, time): phase
        for time, phase in disrupt_phase_dict.items()
    }
    return disrupt_phase_dict

def generate_island_rotation_phase_trajectory(
    trigger_time: float, rot_dur: float, locking_dur: float
):
    # TODO(allenw): we want a rectilinear interpolation scheme.
    rot_phase_dict = {
        0.0: IslandRotationPhase.NONE,
        trigger_time - dt: IslandRotationPhase.NONE,
        trigger_time: IslandRotationPhase.SPAWN,
        trigger_time + dt: IslandRotationPhase.ROTATING,
        trigger_time + rot_dur - dt: IslandRotationPhase.ROTATING,
        trigger_time + rot_dur: IslandRotationPhase.DECELERATING,
        trigger_time + rot_dur + locking_dur - dt: IslandRotationPhase.DECELERATING,
        trigger_time + rot_dur + locking_dur: IslandRotationPhase.LOCKED,
    }

    # Round the times to the nearest time step in the time base.
    rot_phase_dict = {
        find_nearest(time_base, time): phase for time, phase in rot_phase_dict.items()
    }
    return rot_phase_dict

two_one_island = Island(2, 1)

islands = [two_one_island]

W = {island: 0.0 for island in islands}
F = {island: 0.0 for island in islands}
wave_phase={island: 0.0 for island in islands}

initial_state = Tearing.State(W=W, F=F, wave_phase=wave_phase)

q2_rot_freq = 7e3
rot_dur = 1.0
trigger_time = 5.0
disrupt_time = 6.5
dur_tq_to_spike = 1e-3
dur_cq = 10e-3
survival_time = 0.3
locking_dur = 0.2

params = Tearing.Params(
    q2_rot_freq=7e3,  # Hz
    rot_dur=1.0,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(
        disrupt_time, dur_tq_to_spike
    ),
    island_rotation_phase=generate_island_rotation_phase_trajectory(
        trigger_time, rot_dur, locking_dur
    ),
)

In [16]:
import jax
jax.config.update("jax_platforms", "cpu")
tearing_module = Tearing(config=config, islands=islands)

sol_xarray = simulate(tearing_module, time_base, initial_state, params)

In [17]:
from popsim.visualize import visualize_time_series

sol_xarray.to_netcdf("tearing_simulation.nc")
visualize_time_series(sol_xarray, max_cols=2)


BokehModel(combine_events=True, render_bundle={'docs_json': {'f04ee30c-c1db-4632-9578-327c57de8cb5': {'version…

In [5]:
import json

# Get sensor positions
with open("/home/zkeith/proj/ONW/ryan-onsim/Device-description/device_description/SPARC/240510/device.json") as f:
    device_description = json.load(f)

BpLowmnInd = np.where(np.array([x['name'][0:7] for x in device_description['magnetics']['b_field_pol_probe']]) == 'BP-LOMN')[0]
BnLowmnInd = np.where(np.array([x['name'][0:7] for x in device_description['magnetics']['b_field_pol_probe']]) == 'BN-LOMN')[0]

phi_probes = [device_description['magnetics']['b_field_pol_probe'][i]['position']['phi'] for i in BpLowmnInd]
phi_sens = [device_description['magnetics']['b_field_pol_probe'][i]['position']['phi'] for i in BnLowmnInd]

In [6]:
from scipy.interpolate import interp1d

# Get frequency responses
filepath = "../../modules/" 
fname = '21_mode_resp_data.txt'
out = np.loadtxt(filepath + fname, skiprows=1)

freq = out[:,0]
Bp_per_A = out[:,1] # Bp (poloidal field) per Amp of tearing mode current
Br_per_A = out[:,2] # Br (radial field) per Amp of tearing mode current


# we also want to build interpolated functions that can be called for 
# any frequency in order to get the br and bp
func_Bp_per_A = interp1d(freq, Bp_per_A) # name stands for "function, bp per A"
func_Br_per_A = interp1d(freq, Br_per_A) # name stands for "function, br per A"

In [9]:
import xarray as xr
from popsim.modules.tearing import IslandModeNumber
# Get the measured magnetic signal for each b-dot
curPerW = 1e3/1e-2 # 1 kA/cm <- a guess for now

# Set up dataset for magnetic measurement, in Gauss
magnetic_measurements = xr.Dataset()
magnetic_measurements['time'] = sol_xarray.time

poloidal_probe_measures = []
radial_sensor_measures = []
for i, angle in enumerate(phi_probes):

    probe_angle = phi_probes[i]
    sensor_angle = phi_sens[i]

    poloidal_probe_measure = 0
    radial_sensor_measure = 0
    for mode in IslandModeNumber:
        # Get values from tearing simulation
        mode_width = sol_xarray[f"state.W.{str(mode)}"]
        mode_freq = sol_xarray[f"state.F.{str(mode)}"]
        mode_angle = sol_xarray[f"state.wave_phase.{str(mode)}"]

        # Set up conversions for what the b-dots measure in Gauss
        mode_current = mode_width * curPerW
        poloidal_frequency_response = func_Bp_per_A(mode_freq)
        radial_frequency_response = func_Br_per_A(mode_freq)

        if mode == IslandModeNumber.THREE_ONE:
            mode_multiplier = 1
        elif mode == IslandModeNumber.TWO_ONE:
            mode_multiplier = 1
        elif mode == IslandModeNumber.THREE_TWO:
            mode_multiplier = 2

        poloidal_probe_measure += mode_current*poloidal_frequency_response*np.cos(phi_probes[i] - (mode_angle*mode_multiplier))
        radial_sensor_measure += mode_current*radial_frequency_response*np.sin(phi_sens[i] - (mode_angle*mode_multiplier))
    
    poloidal_probe_measures.append(poloidal_probe_measure)
    radial_sensor_measures.append(radial_sensor_measure)

magnetic_measurements['poloidal'] = xr.DataArray(poloidal_probe_measures, dims=['phi_probe', 'time'])
magnetic_measurements['phi_probe'] = phi_probes
magnetic_measurements['radial'] = xr.DataArray(radial_sensor_measures, dims=['phi_sensor', 'time'])
magnetic_measurements['phi_sensor'] = phi_sens


magnetic_measurements.to_netcdf("magnetic_measurements.nc")


In [8]:
# Load 

# Using 12 probe design from 2 years ago
# 8, 211, 91, 271, 288, 171, 228, 351, 131, 48, 328, 311
probe_order_indices = [0, 9, 3, 11, 12, 7, 10, 15, 5, 2, 14, 13]
probe_connection_indices = []
for i in range(len(probe_order_indices)):
    if i != len(probe_order_indices) - 1:
        probe_connection_indices.append([probe_order_indices[i], probe_order_indices[i+1]])
    else:
        probe_connection_indices.append([probe_order_indices[i], probe_order_indices[0]])

included_n_modes = [1, 2] # 3/2, 2/1, 3/1

design_matrix = np.zeros((len(probe_connection_indices), 2*len(included_n_modes)))
for i, connection in enumerate(probe_connection_indices):
    for j, mode in enumerate(included_n_modes):
        design_matrix[i, 2*j] = np.cos(mode*phi_probes[connection[0]]) - np.cos(mode*phi_probes[connection[1]])
        design_matrix[i, 2*j+1] = np.sin(mode*phi_probes[connection[0]]) - np.sin(mode*phi_probes[connection[1]])

connection_measurements = np.zeros((len(probe_connection_indices), len(time_base)))
for i, connection in enumerate(probe_connection_indices):
    connection_measurements[i, :] = poloidal_probe_measures[connection[0]] - poloidal_probe_measures[connection[1]]

#mode_components = np.linalg.lstsq(design_matrix, connection_measurements, rcond=None)[0]
mode_components = np.linalg.pinv(design_matrix) @ connection_measurements

mode_magnitudes = np.zeros((len(included_n_modes), len(time_base)))
for i, mode in enumerate(included_n_modes):
    mode_magnitudes[i, :] = np.sqrt(mode_components[2*i, :]**2 + mode_components[2*i+1, :]**2)

mode_magnitudes_xr = xr.Dataset()
mode_magnitudes_xr['time'] = time_base
mode_magnitudes_xr['mode'] = [f"n={str(mode)}" for mode in included_n_modes]
mode_magnitudes_xr['reconstructed_magnitudes'] = xr.DataArray(mode_magnitudes, dims=['mode', 'time'])

mode_magnitudes_xr.to_netcdf("mode_magnitudes.nc")

